# 08 • Déboguer cinq incidents

`[MÉTA | Formation 4-024 | Niveau Application | TP 08 | Mode CPU local]`

**Objectif :** Identifier une cause, corriger et apporter une preuve.

**Temps indicatif :** Défi J2, trois essais maximum. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** TP 02 à 07.

**Preuves de réussite :** Au moins trois incidents expliqués et deux corrections justifiées.

**Sources :** R02, R03, R04.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [1]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


Moteur disponible : 2.10.0+cpu | Données : /mnt/data/deep_learning_4_024/03_Travaux_pratiques/donnees


## 1. Dossier d’incidents
Les extraits suivants sont volontairement erronés et présentés comme du texte. Ils ne sont pas exécutés. Le correctif doit montrer un protocole, pas seulement supprimer un message d’erreur.

In [2]:
incidents={
'A':"scaler.fit(X_complet) ; split(X_complet)",
'B':"evaluation = modele(images)  # sans modele.eval(), avec Dropout",
'C':"loss = CrossEntropyLoss()(softmax(logits), labels_float)",
'D':"accuracy = 0.99  # seule métrique, 1 % de positifs",
'E':"choisir le meilleur taux en consultant le test après chaque essai"
}
for k,v in incidents.items():print(k,':',v)

A : scaler.fit(X_complet) ; split(X_complet)
B : evaluation = modele(images)  # sans modele.eval(), avec Dropout
C : loss = CrossEntropyLoss()(softmax(logits), labels_float)
D : accuracy = 0.99  # seule métrique, 1 % de positifs
E : choisir le meilleur taux en consultant le test après chaque essai


## 2. Corriger la fuite
Le résultat postérieur à l’examen peut parfaitement prédire la cible parce qu’il la recopie. Ce score spectaculaire est volontairement inutilisable au moment réel de la prédiction.

In [3]:
df,X,y=read_tabular();d=split_tabular();tr,va,te=d['indices']
bon=LogisticRegression(max_iter=300).fit(*d['train'])
fuite=LogisticRegression().fit(df[['resultat_apres_examen']].iloc[tr],y[tr])
print('AP sans fuite :',average_precision_score(y[va],bon.predict_proba(d['validation'][0])[:,1]))
print('AP avec information future interdite :',average_precision_score(y[va],fuite.predict_proba(df[['resultat_apres_examen']].iloc[va])[:,1]))
assert 'resultat_apres_examen' not in [f'v{i:02d}' for i in range(20)]

AP sans fuite : 0.7321897838927701
AP avec information future interdite : 1.0


## 3. Corriger sortie et perte
Pour CrossEntropyLoss : logits de forme (lot, classes) et cibles entières de forme (lot). Le label ne doit pas être une probabilité issue du modèle.

In [4]:
logits=torch.randn(4,3,requires_grad=True)
labels=torch.tensor([0,2,1,0],dtype=torch.long)
perte=nn.CrossEntropyLoss()(logits,labels);perte.backward()
assert tuple(logits.grad.shape)==(4,3) and torch.isfinite(perte)
print('Perte valide :',perte.item())

Perte valide : 1.6286567449569702


## 4. Corriger le mode et le reporting
Réduire l’incertitude technique ne dispense pas de documenter les limites métier.

In [5]:
m=nn.Sequential(nn.Linear(5,8),nn.ReLU(),nn.Dropout(.5),nn.Linear(8,1));x=torch.ones(4,5)
m.eval()
with torch.no_grad():a=m(x);b=m(x)
assert torch.equal(a,b)
yrare=np.array([0]*99+[1]);trivial=np.zeros(100)
print('Majoritaire :',binary_metrics(yrare,trivial,.5))
corrections={'A':'split puis fit scaler uniquement sur train','B':'eval et no_grad pour évaluer','C':'logits et classes entières','D':'matrice, précision, rappel, AP et capacité','E':'choix sur validation ; test ouvert une fois'}
print(json.dumps(corrections,ensure_ascii=False,indent=2));save_result('08_diagnostic',corrections)

Majoritaire : {'seuil': 0.5, 'exactitude': 0.99, 'precision': 0.0, 'rappel': 0.0, 'f1': 0.0, 'roc_auc': 0.5, 'precision_moyenne_AP': 0.01, 'brier': 0.01, 'alertes': 0}
{
  "A": "split puis fit scaler uniquement sur train",
  "B": "eval et no_grad pour évaluer",
  "C": "logits et classes entières",
  "D": "matrice, précision, rappel, AP et capacité",
  "E": "choix sur validation ; test ouvert une fois"
}


PosixPath('/mnt/data/deep_learning_4_024/03_Travaux_pratiques/resultats/08_diagnostic.json')

## 5. Restitution
Pour chaque incident : symptôme observé, cause supposée, test discriminant, correction, preuve, limite. **Barème formatif :** cause 2 points ; correction 2 points ; preuve 1 point, pour chaque incident. Une confusion entre validation et test nécessite une remédiation avant le projet.